In [110]:
!pip install -q ultralytics transformers accelerate gradio yt-dlp
!mkdir -p /kaggle/working/clean_cctv

import os
import glob
import subprocess

subprocess.run('curl -s -L -o /kaggle/working/clean_cctv/store_aisle.mp4 "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/store-aisle-detection.mp4"', shell=True)
subprocess.run('curl -s -L -o /kaggle/working/clean_cctv/street_traffic.mp4 "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"', shell=True)

raw_vtest = "/kaggle/working/clean_cctv/raw_vtest.avi"
fixed_vtest = "/kaggle/working/clean_cctv/hallway_walking.mp4"
if not os.path.exists(fixed_vtest):
    subprocess.run(f'curl -s -L -o {raw_vtest} "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"', shell=True)
    subprocess.run(f'ffmpeg -y -i {raw_vtest} -vcodec libx264 -pix_fmt yuv420p -crf 23 {fixed_vtest}', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if os.path.exists(raw_vtest):
        os.remove(raw_vtest)

all_vids = (
    glob.glob("/kaggle/input/**/*.mp4", recursive=True) +
    glob.glob("/kaggle/input/**/*.avi", recursive=True) +
    glob.glob("/kaggle/working/clean_cctv/*.mp4")
)

print(f"Discovered {len(all_vids)} operational video feeds.")

Discovered 2064 operational video feeds.


In [105]:
import cv2
import math
import traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from transformers import AutoProcessor, AutoModel
from ultralytics import YOLO
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COCO_SURVEILLANCE_CLASSES = [0, 1, 2, 3, 5, 7, 24, 26, 28, 32]

@dataclass
class FrameDetection:
    frame_idx: int
    timestamp: float
    bbox: Tuple[float, float, float, float]
    cls_id: int
    conf: float
    state_8d: np.ndarray

@dataclass
class SpatioTemporalTube:
    track_id: int
    cls_id: int = 0
    detections: List[FrameDetection] = field(default_factory=list)
    cached_crops: List[Tuple[float, float, Image.Image]] = field(default_factory=list)
    semantic_score: float = 0.0
    kinematic_score: float = 0.0
    displacement_score: float = 0.0
    temporal_overlap: float = 0.0
    final_rank_score: float = 0.0

    @property
    def start_time(self) -> float:
        return self.detections[0].timestamp if self.detections else 0.0

    @property
    def end_time(self) -> float:
        return self.detections[-1].timestamp if self.detections else 0.0

    @property
    def num_frames(self) -> int:
        return len(self.detections)


class KinematicStateEngine:
    def __init__(self, fps: float = 30.0, ema_alpha: float = 0.30):
        self.dt = 1.0 / max(fps, 1.0)
        self.alpha = ema_alpha
        self.tracks: Dict[int, Dict[str, float]] = {}

    def compute_state(self, track_id: int, bbox: Tuple[float, float, float, float], step_frames: int = 1) -> np.ndarray:
        x1, y1, x2, y2 = bbox
        raw_xc = (x1 + x2) / 2.0
        raw_yc = (y1 + y2) / 2.0
        w = max(1.0, x2 - x1)
        h = max(1.0, y2 - y1)
        eff_dt = self.dt * max(1, step_frames)

        if track_id not in self.tracks:
            self.tracks[track_id] = {
                'xc': raw_xc, 'yc': raw_yc,
                'vx': 0.0, 'vy': 0.0,
                'ax': 0.0, 'ay': 0.0
            }
            return np.array([raw_xc, raw_yc, w, h, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)

        prev = self.tracks[track_id]
        xc = self.alpha * raw_xc + (1.0 - self.alpha) * prev['xc']
        yc = self.alpha * raw_yc + (1.0 - self.alpha) * prev['yc']

        raw_vx = (xc - prev['xc']) / eff_dt
        raw_vy = (yc - prev['yc']) / eff_dt
        vx = self.alpha * raw_vx + (1.0 - self.alpha) * prev['vx']
        vy = self.alpha * raw_vy + (1.0 - self.alpha) * prev['vy']

        raw_ax = (vx - prev['vx']) / eff_dt
        raw_ay = (vy - prev['vy']) / eff_dt
        ax = self.alpha * raw_ax + (1.0 - self.alpha) * prev['ax']
        ay = self.alpha * raw_ay + (1.0 - self.alpha) * prev['ay']

        self.tracks[track_id] = {'xc': xc, 'yc': yc, 'vx': vx, 'vy': vy, 'ax': ax, 'ay': ay}
        return np.array([xc, yc, w, h, vx, vy, ax, ay], dtype=np.float32)

In [106]:
class KinematicPathBiGRU(nn.Module):
    def __init__(self, input_dim: int = 8, hidden_dim: int = 64, embed_dim: int = 128):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = self.norm(x)
        gru_out, _ = self.gru(x_norm)
        pooled = torch.mean(gru_out, dim=1)
        return F.normalize(self.proj(pooled), p=2, dim=-1)


class MultiModalGroundingBackbone:
    def __init__(self, device: torch.device):
        self.device = device
        self.dtype = torch.float16 if device.type == "cuda" else torch.float32
        self.model_id = "google/siglip-base-patch16-384"
        self.processor = AutoProcessor.from_pretrained(self.model_id)
        self.siglip = AutoModel.from_pretrained(self.model_id, torch_dtype=self.dtype).to(device).eval()
        self.kinematic_gru = KinematicPathBiGRU().to(device).eval()

    def _extract_tensor(self, model_output) -> torch.Tensor:
        if isinstance(model_output, torch.Tensor):
            return model_output.float()
        if hasattr(model_output, "pooler_output") and model_output.pooler_output is not None:
            return model_output.pooler_output.float()
        if hasattr(model_output, "last_hidden_state"):
            return model_output.last_hidden_state[:, 0, :].float()
        return (model_output[1] if len(model_output) > 1 else model_output[0]).float()

    def encode_text(self, texts: List[str]) -> torch.Tensor:
        inputs = self.processor(text=texts, padding="max_length", return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.siglip.get_text_features(**inputs)
            tensor = self._extract_tensor(out)
            return F.normalize(tensor, p=2, dim=-1)

    def encode_images(self, pil_images: List[Image.Image], batch_size: int = 16) -> torch.Tensor:
        all_embeds = []
        for i in range(0, len(pil_images), batch_size):
            batch = pil_images[i:i + batch_size]
            inputs = self.processor(images=batch, return_tensors="pt").to(self.device)
            if self.device.type == "cuda":
                inputs["pixel_values"] = inputs["pixel_values"].to(self.dtype)
            with torch.no_grad():
                out = self.siglip.get_image_features(**inputs)
                tensor = self._extract_tensor(out)
                all_embeds.append(F.normalize(tensor, p=2, dim=-1))
        return torch.cat(all_embeds, dim=0)

    def calibrated_similarity(self, img_embeds: torch.Tensor, txt_embeds: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            dots = torch.matmul(img_embeds, txt_embeds.T)
            if hasattr(self.siglip, "logit_scale") and hasattr(self.siglip, "logit_bias"):
                scale = self.siglip.logit_scale.exp().float()
                bias = self.siglip.logit_bias.float()
                return torch.sigmoid(dots * scale + bias)
            return torch.sigmoid(dots * 10.0)

    @staticmethod
    def extract_letterbox_crop(frame_rgb: np.ndarray, bbox: Tuple[float, float, float, float], pad_ratio: float = 0.22) -> Optional[Image.Image]:
        h_img, w_img = frame_rgb.shape[:2]
        x1, y1, x2, y2 = bbox
        bw, bh = x2 - x1, y2 - y1
        if bw < 14 or bh < 22:
            return None

        px = bw * pad_ratio
        py = bh * pad_ratio
        nx1 = int(max(0, x1 - px))
        ny1 = int(max(0, y1 - py))
        nx2 = int(min(w_img, x2 + px))
        ny2 = int(min(h_img, y2 + py))

        crop = frame_rgb[ny1:ny2, nx1:nx2]
        ch, cw = crop.shape[:2]
        if ch < 16 or cw < 16:
            return None

        side = max(ch, cw)
        canvas = np.full((side, side, 3), 114, dtype=np.uint8)
        y_off = (side - ch) // 2
        x_off = (side - cw) // 2
        canvas[y_off:y_off + ch, x_off:x_off + cw] = crop
        return Image.fromarray(canvas)

In [107]:
class OmniStreamPipeline:
    def __init__(self, device: torch.device, min_tube_frames: int = 4):
        self.device = device
        self.min_tube_frames = min_tube_frames
        self.yolo = YOLO("yolov8m.pt")
        self.backbone = MultiModalGroundingBackbone(device)

    @staticmethod
    def _is_kinetic_query(query: str) -> bool:
        kinetic_tokens = {
            "walk", "walking", "run", "running", "move", "moving", "drive", "driving",
            "ride", "riding", "carry", "carrying", "open", "opening", "close", "closing",
            "enter", "entering", "exit", "exiting", "fight", "fighting", "cross", "crossing",
            "load", "loading", "unload", "unloading", "get", "getting", "push", "pull", "throw"
        }
        words = set(query.lower().replace(",", " ").replace(".", " ").split())
        return len(words.intersection(kinetic_tokens)) > 0

    @staticmethod
    def _compute_dynamic_window(timestamps: List[float], curve: np.ndarray, peak_idx: int, duration: float) -> Tuple[float, float]:
        peak_val = curve[peak_idx]
        base_val = np.percentile(curve, 25)
        thresh = base_val + 0.45 * (peak_val - base_val)

        left = peak_idx
        while left > 0 and curve[left - 1] >= thresh:
            left -= 1

        right = peak_idx
        while right < len(curve) - 1 and curve[right + 1] >= thresh:
            right += 1

        t_start = timestamps[left]
        t_end = timestamps[right]

        if (t_end - t_start) < 2.4:
            center = timestamps[peak_idx]
            t_start = max(0.0, center - 1.2)
            t_end = min(duration, center + 1.2)
        else:
            t_start = max(0.0, t_start - 0.3)
            t_end = min(duration, t_end + 0.3)

        return t_start, t_end

    def process_video(self, video_path: str, query: str, max_analysis_sec: float = 45.0) -> Dict:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = min(max_analysis_sec, max(0.1, total_frames / fps))
        max_frame_limit = int(duration * fps)

        track_stride = max(1, int(round(fps / 12.0)))
        vlm_stride = max(1, int(round(fps / 4.0)))

        kinematics = KinematicStateEngine(fps=fps)
        tubes: Dict[int, SpatioTemporalTube] = {}

        sampled_frames_pil = []
        sampled_timestamps = []
        sampled_motion_energy = []
        motion_zones = {}

        prev_gray = None
        frame_idx = 0

        self.yolo.predictor = None

        while cap.isOpened() and frame_idx < max_frame_limit:
            ret, frame = cap.read()
            if not ret:
                break

            timestamp = frame_idx / fps

            if frame_idx % track_stride == 0:
                results = self.yolo.track(
                    frame,
                    persist=True,
                    tracker="bytetrack.yaml",
                    classes=COCO_SURVEILLANCE_CLASSES,
                    conf=0.25,
                    iou=0.5,
                    verbose=False
                )

                rgb_curr = None
                if results and results[0].boxes and results[0].boxes.id is not None:
                    boxes = results[0].boxes.xyxy.cpu().numpy()
                    track_ids = results[0].boxes.id.int().cpu().numpy()
                    clss = results[0].boxes.cls.int().cpu().numpy()
                    confs = results[0].boxes.conf.cpu().numpy()

                    for bbox, tid, cid, cf in zip(boxes, track_ids, clss, confs):
                        tid_int = int(tid)
                        state_8d = kinematics.compute_state(tid_int, tuple(bbox), step_frames=track_stride)
                        det = FrameDetection(frame_idx, timestamp, tuple(bbox), int(cid), float(cf), state_8d)

                        if tid_int not in tubes:
                            tubes[tid_int] = SpatioTemporalTube(track_id=tid_int, cls_id=int(cid))
                        tube = tubes[tid_int]
                        tube.detections.append(det)

                        if len(tube.detections) % 2 == 1:
                            if rgb_curr is None:
                                rgb_curr = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                            crop_img = self.backbone.extract_letterbox_crop(rgb_curr, tuple(bbox))
                            if crop_img is not None:
                                area_score = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]) * float(cf)
                                tube.cached_crops.append((timestamp, area_score, crop_img))
                                if len(tube.cached_crops) > 18:
                                    tube.cached_crops.sort(key=lambda x: x[1], reverse=True)
                                    tube.cached_crops = tube.cached_crops[:12]

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray_small = cv2.resize(gray, (320, 180))
            gray_blur = cv2.GaussianBlur(gray_small, (5, 5), 0)

            if prev_gray is not None:
                diff = cv2.absdiff(gray_blur, prev_gray)
                energy = float(np.mean(diff))
                _, thresh = cv2.threshold(diff, 22, 255, cv2.THRESH_BINARY)
                contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    largest = max(contours, key=cv2.contourArea)
                    if cv2.contourArea(largest) > 110:
                        mx, my, mw, mh = cv2.boundingRect(largest)
                        sx, sy = width / 320.0, height / 180.0
                        motion_zones[frame_idx] = (
                            int(mx * sx), int(my * sy),
                            int((mx + mw) * sx), int((my + mh) * sy)
                        )
            else:
                energy = 0.0
            prev_gray = gray_blur

            if frame_idx % vlm_stride == 0:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                sampled_frames_pil.append(Image.fromarray(rgb_frame))
                sampled_timestamps.append(timestamp)
                sampled_motion_energy.append(energy)

            frame_idx += 1

        cap.release()

        if not sampled_frames_pil:
            return {"error": "Unable to decode frames from video."}

        valid_tubes = {tid: t for tid, t in tubes.items() if t.num_frames >= self.min_tube_frames}
        clean_q = query.strip()
        is_kinetic = self._is_kinetic_query(clean_q)

        prompts = [
            clean_q,
            f"surveillance view of {clean_q}",
            "empty background pavement wall parked stationary"
        ]
        text_embeds = self.backbone.encode_text(prompts)
        pos_embed = F.normalize(text_embeds[0:2].mean(dim=0, keepdim=True), p=2, dim=-1)
        neg_embed = text_embeds[2:3]

        frame_embeds = self.backbone.encode_images(sampled_frames_pil)
        pos_probs = self.backbone.calibrated_similarity(frame_embeds, pos_embed).squeeze(-1).cpu().numpy()
        neg_probs = self.backbone.calibrated_similarity(frame_embeds, neg_embed).squeeze(-1).cpu().numpy()
        net_sem = pos_probs - 0.35 * neg_probs

        s_min, s_max = net_sem.min(), net_sem.max()
        norm_sem = (net_sem - s_min) / (s_max - s_min + 1e-6)

        mot_arr = np.array(sampled_motion_energy, dtype=np.float32)
        norm_mot = mot_arr / (np.percentile(mot_arr, 95) + 1e-6)
        norm_mot = np.clip(norm_mot, 0.0, 1.0)

        w_sem = 0.60 if is_kinetic else 0.80
        w_mot = 0.40 if is_kinetic else 0.20
        temporal_curve = w_sem * norm_sem + w_mot * norm_mot

        k_win = min(5, len(temporal_curve))
        kernel = np.array([0.15, 0.2, 0.3, 0.2, 0.15]) if k_win == 5 else np.ones(k_win) / k_win
        smoothed_curve = np.convolve(temporal_curve, kernel, mode='same')

        peak_idx = int(np.argmax(smoothed_curve))
        peak_time = sampled_timestamps[peak_idx]
        t_start, t_end = self._compute_dynamic_window(sampled_timestamps, smoothed_curve, peak_idx, duration)

        candidate_ids = [
            tid for tid, t in valid_tubes.items()
            if t.start_time <= t_end and t.end_time >= t_start and len(t.cached_crops) > 0
        ]

        for tid in candidate_ids:
            tube = valid_tubes[tid]
            win_dets = [d for d in tube.detections if t_start <= d.timestamp <= t_end]
            if not win_dets:
                win_dets = tube.detections

            speeds = [math.hypot(d.state_8d[4], d.state_8d[5]) for d in win_dets]
            tube.kinematic_score = float(np.percentile(speeds, 75)) if speeds else 0.0

            if len(win_dets) >= 2:
                dx = win_dets[-1].state_8d[0] - win_dets[0].state_8d[0]
                dy = win_dets[-1].state_8d[1] - win_dets[0].state_8d[1]
                diag = math.hypot(win_dets[0].state_8d[2], win_dets[0].state_8d[3]) + 1e-4
                tube.displacement_score = float(math.hypot(dx, dy) / diag)
            else:
                tube.displacement_score = 0.0

            overlap_dur = max(0.0, min(tube.end_time, t_end) - max(tube.start_time, t_start))
            tube.temporal_overlap = min(1.0, overlap_dur / max(0.5, (t_end - t_start)))

            sorted_crops = sorted(tube.cached_crops, key=lambda x: abs(x[0] - peak_time))
            selected_imgs = [c[2] for c in sorted_crops[:5]]

            crop_embeds = self.backbone.encode_images(selected_imgs)
            c_pos = self.backbone.calibrated_similarity(crop_embeds, pos_embed).squeeze(-1).mean().item()
            c_neg = self.backbone.calibrated_similarity(crop_embeds, neg_embed).squeeze(-1).mean().item()
            tube.semantic_score = float(c_pos - 0.30 * c_neg)

        scored_candidates = [valid_tubes[tid] for tid in candidate_ids]
        best_tube_id = None
        ranking_log = []

        if scored_candidates:
            sem_vals = np.array([t.semantic_score for t in scored_candidates], dtype=np.float32)
            kin_vals = np.array([t.kinematic_score for t in scored_candidates], dtype=np.float32)
            disp_vals = np.array([t.displacement_score for t in scored_candidates], dtype=np.float32)
            ovl_vals = np.array([t.temporal_overlap for t in scored_candidates], dtype=np.float32)

            if len(sem_vals) > 1 and sem_vals.max() > sem_vals.min():
                sem_norm = (sem_vals - sem_vals.min()) / (sem_vals.max() - sem_vals.min() + 1e-6)
            else:
                sem_norm = np.ones_like(sem_vals)

            kin_norm = kin_vals / (kin_vals.max() + 1e-6)
            disp_norm = np.clip(disp_vals / ( np.percentile(disp_vals, 90) + 1e-6 ), 0.0, 1.0)

            for i, t in enumerate(scored_candidates):
                if is_kinetic:
                    motion_factor = 0.55 * disp_norm[i] + 0.45 * kin_norm[i]
                    t.final_rank_score = float(0.55 * sem_norm[i] + 0.35 * motion_factor + 0.10 * ovl_vals[i])
                else:
                    t.final_rank_score = float(0.80 * sem_norm[i] + 0.10 * disp_norm[i] + 0.10 * ovl_vals[i])

            scored_candidates.sort(key=lambda t: t.final_rank_score, reverse=True)
            best_tube = scored_candidates[0]
            best_tube_id = best_tube.track_id

            best_dets = [d for d in best_tube.detections if t_start <= d.timestamp <= t_end]
            if best_dets and is_kinetic:
                peak_det = max(best_dets, key=lambda d: math.hypot(d.state_8d[4], d.state_8d[5]))
                peak_time = peak_det.timestamp
                t_start = max(0.0, min(t_start, peak_time - 1.2))
                t_end = min(duration, max(t_end, peak_time + 1.2))

            for t in scored_candidates[:4]:
                ranking_log.append(
                    f"ID#{t.track_id}[R:{t.final_rank_score:.2f}|S:{t.semantic_score:.2f}|D:{t.displacement_score:.1f}]"
                )

        confidence = float(np.clip(smoothed_curve[peak_idx], 0.0, 1.0))

        return {
            "query": query,
            "peak_time": peak_time,
            "confidence": confidence,
            "interval": (t_start, t_end),
            "total_tubes": len(valid_tubes),
            "target_tube_id": best_tube_id,
            "target_tube": valid_tubes.get(best_tube_id, None),
            "motion_zones": motion_zones,
            "ranking_summary": "  ".join(ranking_log) if ranking_log else "Scene-Level Activity",
            "fps": fps,
            "width": width,
            "height": height
        }

In [112]:
def draw_tactical_box(img: np.ndarray, x1: int, y1: int, x2: int, y2: int, color: Tuple[int, int, int], thickness: int = 2):
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 1, cv2.LINE_AA)
    corner_len = min(18, (x2 - x1) // 3, (y2 - y1) // 3)
    if corner_len < 4:
        return
    cv2.line(img, (x1, y1), (x1 + corner_len, y1), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x1, y1), (x1, y1 + corner_len), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x2, y1), (x2 - corner_len, y1), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x2, y1), (x2, y1 + corner_len), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x1, y2), (x1 + corner_len, y2), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x1, y2), (x1, y2 - corner_len), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x2, y2), (x2 - corner_len, y2), color, thickness + 1, cv2.LINE_AA)
    cv2.line(img, (x2, y2), (x2, y2 - corner_len), color, thickness + 1, cv2.LINE_AA)


def render_tactical_hud_moment(video_path: str, result: Dict, output_path: str) -> str:
    cap = cv2.VideoCapture(video_path)
    fps = result['fps']
    w, h = result['width'], result['height']
    t_start, t_end = result['interval']
    query = result['query']
    target_id = result['target_tube_id']
    tube = result['target_tube']
    motion_zones = result.get('motion_zones', {})

    temp_raw = "/kaggle/working/hud_intermediate.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(temp_raw, fourcc, fps, (w, h))

    det_frames = sorted([d.frame_idx for d in tube.detections]) if tube else []
    det_map = {d.frame_idx: d for d in tube.detections} if tube else {}
    trajectory_trail = []

    start_frame = int(t_start * fps)
    end_frame = int(t_end * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    curr_frame = start_frame

    while cap.isOpened() and curr_frame <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = curr_frame / fps

        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 42), (12, 12, 18), -1)
        frame = cv2.addWeighted(overlay, 0.82, frame, 0.18, 0)

        cv2.putText(frame, f"OMNISTREAM V3.0 | QUERY: {query.upper()[:38]}", (14, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 240, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"T: {timestamp:.2f}s | CONF: {result['confidence']*100:.1f}%", ( max(10, w - 235), 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (50, 255, 100), 2, cv2.LINE_AA)

        active_det = None
        if curr_frame in det_map:
            active_det = det_map[curr_frame]
        elif det_frames:
            nearest_f = min(det_frames, key=lambda f: abs(f - curr_frame))
            if abs(nearest_f - curr_frame) <= 4:
                active_det = det_map[nearest_f]

        if active_det is not None:
            x1, y1, x2, y2 = map(int, active_det.bbox)
            xc, yc, bw, bh, vx, vy, ax, ay = active_det.state_8d
            speed = math.hypot(vx, vy)
            accel = math.hypot(ax, ay)

            trajectory_trail.append((int(xc), int(yc)))
            if len(trajectory_trail) > 35:
                trajectory_trail.pop(0)

            for i in range(1, len(trajectory_trail)):
                alpha_ratio = i / len(trajectory_trail)
                thickness = max(1, int(3 * alpha_ratio))
                cv2.line(frame, trajectory_trail[i - 1], trajectory_trail[i], (0, int(180 * alpha_ratio), 255), thickness, cv2.LINE_AA)

            if speed > 8.0:
                end_vx = int(xc + np.clip(vx * 0.25, -45, 45))
                end_vy = int(yc + np.clip(vy * 0.25, -45, 45))
                cv2.arrowedLine(frame, (int(xc), int(yc)), (end_vx, end_vy), (0, 255, 255), 2, cv2.LINE_AA, tipLength=0.3)

            draw_tactical_box(frame, x1, y1, x2, y2, (0, 40, 255), thickness=2)
            label = f"ID:{target_id} | V:{speed:.0f}px/s | A:{accel:.0f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.44, 1)
            ly1 = max(44, y1 - th - 10)
            cv2.rectangle(frame, (x1, ly1), (x1 + tw + 10, ly1 + th + 8), (0, 40, 255), -1)
            cv2.putText(frame, label, (x1 + 5, ly1 + th + 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.44, (255, 255, 255), 1, cv2.LINE_AA)

        elif curr_frame in motion_zones:
            mx1, my1, mx2, my2 = motion_zones[curr_frame]
            draw_tactical_box(frame, mx1, my1, mx2, my2, (0, 165, 255), thickness=2)
            cv2.putText(frame, "KINETIC SALIENCY REGION", (mx1, max(55, my1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 165, 255), 1, cv2.LINE_AA)

        writer.write(frame)
        curr_frame += 1

    cap.release()
    writer.release()

    subprocess.run(
        f"ffmpeg -y -i {temp_raw} -vcodec libx264 -pix_fmt yuv420p -crf 21 {output_path}",
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
    )
    return output_path

/usr/local/lib/python3.13/dist-packages/gradio/components/video.py:456: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(


In [109]:
import os
import glob
import traceback
import subprocess
import gradio as gr

engine = OmniStreamPipeline(device=device, min_tube_frames=4)

raw_candidates = (
    glob.glob("/kaggle/working/clean_cctv/*.mp4") +
    glob.glob("/kaggle/input/**/*.mp4", recursive=True) +
    glob.glob("/kaggle/input/**/*.avi", recursive=True)
)

video_choices = {}
for p in raw_candidates[:60]:
    bname = os.path.basename(p)
    parent = os.path.basename(os.path.dirname(p))
    lbl = f"{parent}/{bname}" if bname in video_choices else bname
    video_choices[lbl] = p

default_label = list(video_choices.keys())[0] if video_choices else None

def prepare_web_video(raw_path: str) -> str:
    if not raw_path or not os.path.exists(raw_path):
        return None
    
    web_ready_path = "/kaggle/working/active_preview.mp4"
    if raw_path.lower().endswith(".mp4"):
        check_cmd = f"ffprobe -v error -select_streams v:0 -show_entries stream=codec_name -of default=noprint_wrappers=1:nokey=1 '{raw_path}'"
        codec = subprocess.getoutput(check_cmd).strip().lower()
        if "h264" in codec:
            return raw_path

    cmd = f"ffmpeg -y -i '{raw_path}' -t 20 -vcodec libx264 -pix_fmt yuv420p -preset ultrafast -crf 23 '{web_ready_path}'"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return web_ready_path if os.path.exists(web_ready_path) else raw_path

def on_select_video(selected_name: str):
    real_path = video_choices.get(selected_name)
    return prepare_web_video(real_path)

def fetch_url(url: str):
    if not url or not url.strip():
        return None
    out_path = "/kaggle/working/active_preview.mp4"
    cmd = f"yt-dlp -f 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4' --download-sections '*00:00-00:20' -o '{out_path}' '{url.strip()}'"
    subprocess.run(cmd, shell=True, check=True)
    return out_path

def execute_grounding(video_file, search_query):
    if not video_file or not str(search_query).strip():
        return None, "Please select an active video stream and provide a query."

    try:
        result = engine.process_video(video_file, search_query)
        if "error" in result:
            return None, f"Pipeline Error: {result['error']}"

        output_clip = "/kaggle/working/omnistream_grounded_clip.mp4"
        render_tactical_hud_moment(video_file, result, output_clip)

        t_start, t_end = result['interval']
        target_str = f"Tube ID #{result['target_tube_id']}" if result['target_tube_id'] is not None else "Global Grounded Saliency"

        report = (
            "═══════════════════════════════════════════════════════════════\n"
            "       OMNISTREAM ENGINE V3.0 TELEMETRY GROUNDING REPORT       \n"
            "═══════════════════════════════════════════════════════════════\n"
            f"• Semantic Query        : \"{result['query']}\"\n"
            f"• Grounded Window       : [{t_start:.2f}s  ──►  {t_end:.2f}s]\n"
            f"• Peak Action Moment    : {result['peak_time']:.2f}s\n"
            f"• Visual-Semantic Score : {result['confidence'] * 100:.2f}%\n"
            f"• Active Scene Tubes    : {result['total_tubes']} verified tracks\n"
            f"• Localized Entity      : {target_str}\n"
            f"• Candidate Log         : {result['ranking_summary']}\n"
            "• Mathematical Engine   : 8D Analytical EMA Kinematics Active\n"
            "═══════════════════════════════════════════════════════════════"
        )
        return output_clip, report

    except Exception as e:
        err_details = traceback.format_exc()
        return None, f"Runtime Exception:\n{str(e)}\n\n{err_details[-500:]}"

initial_preview = prepare_web_video(video_choices.get(default_label)) if default_label else None

with gr.Blocks(title="OmniStream Engine V3.0") as demo:
    gr.Markdown("# 🛰️ OmniStream Engine V3.0: High-Precision Spatio-Temporal Grounding")

    with gr.Row():
        with gr.Column(scale=1):
            dropdown_picker = gr.Dropdown(
                label="Select Surveillance Stream (Dataset / Clean Feeds):",
                choices=list(video_choices.keys()),
                value=default_label
            )
            url_box = gr.Textbox(label="Or Remote Video URL (Optional):", placeholder="https://...")
            fetch_btn = gr.Button("Fetch Stream", variant="secondary")

            active_video = gr.Video(label="Active Video Stream", value=initial_preview, sources=["upload"])
            query_box = gr.Textbox(
                label="Natural Language Query",
                value="person walking"
            )
            submit_btn = gr.Button("⚡ Execute Spatio-Temporal Grounding", variant="primary")

        with gr.Column(scale=1):
            output_video = gr.Video(label="Grounded Moment Clip (Tactical HUD)")
            telemetry_console = gr.Textbox(label="Telemetry & Candidate Diagnostics", lines=13, interactive=False)

    dropdown_picker.change(fn=on_select_video, inputs=[dropdown_picker], outputs=[active_video])
    fetch_btn.click(fn=fetch_url, inputs=[url_box], outputs=[active_video])
    submit_btn.click(fn=execute_grounding, inputs=[active_video, query_box], outputs=[output_video, telemetry_console])

demo.launch(share=True, allowed_paths=["/kaggle"])

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  814MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7881
* Running on public URL: https://61a39a0806184a3726.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2712, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2378, in postprocess_data
    await processing_utils.async_move_files_to_cache(
    ...<3 lines>...
    )
  File "/usr/local/lib/python3.13/dist-package